# QuantiPhy — Stage 2: VLM semantic plan + landmark selection

Notebook này chạy trong **runtime T4 mới**. Nó không tải GroundingDINO hoặc SAM2 weights; chỉ đọc video, parent masks và plans đã lưu bởi Stage 1. VRAM được dành cho VLM 7–8B lượng tử 4-bit.


## 0. Mount Drive và cấu hình handoff


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/quantiphy_baseline')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/quantiphy_artifacts/stage1')
TEST_VIDEO_ID = 'simulation_0009'
TEST_QA_ID = '1095'

track_json = ARTIFACT_ROOT / 'tracks' / f'{TEST_VIDEO_ID}.json'
group_json = ARTIFACT_ROOT / 'handoff' / TEST_VIDEO_ID / 'group.json'
assert (PROJECT_ROOT / 'src/quantiphy_baseline').exists(), 'Sửa PROJECT_ROOT'
assert track_json.exists(), 'Chưa có artifact Stage 1: ' + str(track_json)
assert group_json.exists(), 'Chưa có group handoff: ' + str(group_json)
%cd {PROJECT_ROOT}


## 1. Cài dependency VLM — không cài/chạy GroundingDINO hoặc SAM2


In [ ]:
%pip install -q "transformers>=5.3,<6" "accelerate>=1.0" bitsandbytes safetensors
%pip install -q opencv-python Pillow numpy matplotlib


In [ ]:
import sys, torch, transformers
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('GPU         :', torch.cuda.get_device_name(0))
print('VRAM        : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 2**30))
print('Transformers:', transformers.__version__)


## 2. Chọn VLM

- `best_t4`: Qwen3-VL-8B, ưu tiên để test spatial reasoning/grounding.
- `stable_t4`: Qwen2.5-VL-7B, integration trưởng thành hơn.
- `fast_t4`: Qwen2.5-VL-3B, dùng khi 8B OOM hoặc cần debug nhanh.

Không cần vLLM server trong notebook này; Transformers + bitsandbytes 4-bit ít overhead hơn cho một T4.


In [ ]:
MODEL_PRESETS = {
    'best_t4': 'Qwen/Qwen3-VL-8B-Instruct',
    'stable_t4': 'Qwen/Qwen2.5-VL-7B-Instruct',
    'fast_t4': 'Qwen/Qwen2.5-VL-3B-Instruct',
}
MODEL_PRESET = 'best_t4'
MODEL_ID = MODEL_PRESETS[MODEL_PRESET]
MAX_OVERLAY_SIDE = 1024  # giảm visual tokens/VRAM; labels vẫn đủ rõ
print('Selected:', MODEL_ID)


## 3. Load Stage 1 artifacts, không chạy lại tracker


In [ ]:
import json
from quantiphy_baseline.measurement_plan import MeasurementPlan
from quantiphy_baseline.vision.pipeline import load_track_masks_artifact
from quantiphy_baseline.vision.video_io import load_video_pil

saved_result = json.loads(track_json.read_text(encoding='utf-8'))
group = json.loads(group_json.read_text(encoding='utf-8'))
saved_plans = [MeasurementPlan.from_dict(item) for item in saved_result['measurement_plans']]
saved_plans = [plan for plan in saved_plans if plan.qa_id == TEST_QA_ID]
assert saved_plans, TEST_QA_ID
tracks = load_track_masks_artifact(saved_result)
video_path = Path(saved_result['video_path'])
frames, decoded_fps = load_video_pil(video_path)
fps = float(saved_result.get('fps') or decoded_fps)
question = next(item for item in group['questions'] if str(item['qa_id']) == TEST_QA_ID)
print('Question:', question['raw_question'])
print('Tracks  :', {key: [x.track_id for x in value] for key, value in tracks.items()})
print('Frames  :', len(frames), 'FPS:', fps)


## 4. Load VLM 4-bit trên T4


In [ ]:
from transformers import BitsAndBytesConfig, pipeline

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,  # T4: FP16, không dùng BF16
    bnb_4bit_use_double_quant=True,
)
vlm_pipe = pipeline(
    'image-text-to-text',
    model=MODEL_ID,
    device_map='auto',
    model_kwargs={
        'quantization_config': quantization,
        'dtype': torch.float16,
    },
)
print('Allocated: %.2f GB' % (torch.cuda.memory_allocated() / 2**30))
print('Reserved : %.2f GB' % (torch.cuda.memory_reserved() / 2**30))


## 5. Adapter local VLM cho semantic planner và candidate selector


In [ ]:
import re, uuid
from typing import Any
from PIL import Image
from quantiphy_baseline.vision.vlm_selector import render_candidate_overlay

VLM_DEBUG_DIR = ARTIFACT_ROOT / 'stage2_vlm' / TEST_VIDEO_ID / 'overlays'
VLM_DEBUG_DIR.mkdir(parents=True, exist_ok=True)

def generated_text(output):
    value = output[0].get('generated_text', output[0]) if isinstance(output, list) else output
    if isinstance(value, list):
        value = value[-1].get('content', value[-1])
    if isinstance(value, list):
        value = ''.join(str(item.get('text', '')) if isinstance(item, dict) else str(item)
                        for item in value)
    return str(value)

def extract_json(text):
    text = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        decoder = json.JSONDecoder()
        objects = []
        for match in re.finditer(r'\{', text):
            try:
                value, _ = decoder.raw_decode(text[match.start():])
                if isinstance(value, dict):
                    objects.append(value)
            except json.JSONDecodeError:
                pass
        if not objects:
            raise ValueError('Không tìm thấy JSON trong output: ' + text[:500])
        preferred = [obj for obj in objects if 'operands' in obj or 'selected_candidate' in obj]
        return (preferred or objects)[-1]

def run_vlm(messages, max_new_tokens):
    output = vlm_pipe(
        text=messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    return generated_text(output)

class LocalChatTransport:
    def __call__(self, payload: dict[str, Any]) -> dict[str, Any]:
        text = run_vlm(payload['messages'], max_new_tokens=1200)
        return extract_json(text)

class LocalVLMSelector:
    def select(self, frame, parent_mask, candidates, prompt, reference_mask=None):
        overlay = render_candidate_overlay(frame, parent_mask, candidates, reference_mask)
        if max(overlay.size) > MAX_OVERLAY_SIDE:
            overlay.thumbnail((MAX_OVERLAY_SIDE, MAX_OVERLAY_SIDE), Image.Resampling.LANCZOS)
        image_path = VLM_DEBUG_DIR / f'{uuid.uuid4().hex}.jpg'
        overlay.save(image_path, quality=94)
        valid_ids = [item.candidate_id for item in candidates]
        instruction = (
            f'{prompt} Choose exactly one candidate from {valid_ids}. '
            'Green is the parent mask; cyan is the reference mask. '
            'Return JSON only: {\"selected_candidate\":\"ID\",\"confidence\":0.0}.'
        )
        messages = [{'role': 'user', 'content': [
            {'type': 'image', 'url': str(image_path)},
            {'type': 'text', 'text': instruction},
        ]}]
        raw = run_vlm(messages, max_new_tokens=128)
        parsed = extract_json(raw)
        selected = str(parsed['selected_candidate'])
        if selected not in valid_ids:
            raise ValueError(f'VLM chọn {selected}; candidates={valid_ids}; raw={raw}')
        confidence = max(0.0, min(1.0, float(parsed.get('confidence', 0.5))))
        return selected, confidence, {
            'model': MODEL_ID, 'raw_output': raw, 'overlay_path': str(image_path)
        }

vlm_selector = LocalVLMSelector()


## 6. Test LLM semantic landmark plan bằng chính VLM


In [ ]:
from quantiphy_baseline.planning import LLMSemanticPlanner
from quantiphy_baseline.measurement_plan import build_measurement_plan

semantic_planner = LLMSemanticPlanner(
    model=MODEL_ID,
    transport=LocalChatTransport(),
    cache_path=ARTIFACT_ROOT / 'stage2_vlm' / 'semantic_plan_cache.jsonl',
)
semantic_plan = semantic_planner.plan(question)  # strict: lỗi thì cell dừng, không legacy fallback
llm_plan = build_measurement_plan(question, semantic_plan=semantic_plan)
display(llm_plan.to_dict())

available_tracks = set(tracks)
required_tracks = {operand.tracking_key for operand in llm_plan.operands}
missing_tracks = required_tracks - available_tracks
print('Required :', required_tracks)
print('Available:', available_tracks)
print('Missing  :', missing_tracks)
if missing_tracks:
    print('LLM plan đã yêu cầu parent chưa track. Resolver sẽ dùng saved plan cho test visual này.')
    active_plan = saved_plans[0]
else:
    active_plan = llm_plan


## 7. Xem candidates trước khi hỏi VLM


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quantiphy_baseline.vision.landmark_candidates import generate_landmark_candidates

outer_operand = next(
    operand for operand in active_plan.operands if operand.landmark.kind == 'outer_end'
)
frame_idx = round(float(active_plan.temporal.get('time_s') or 0) * fps)
parent_track = tracks[outer_operand.tracking_key][0]
parent_mask = parent_track.masks[frame_idx].astype(bool)
reference_id = (outer_operand.landmark.selector or {}).get('reference_entity_id')
reference_track = tracks.get(reference_id, [None])[0] if reference_id else None
reference_mask = (reference_track.masks[frame_idx].astype(bool)
                  if reference_track is not None else None)
candidates = generate_landmark_candidates(
    parent_mask,
    outer_operand.landmark.candidate_generator,
    outer_operand.landmark.geometry_type,
    selector_operator=(outer_operand.landmark.selector or {}).get('operator'),
    reference_mask=reference_mask,
)
preview = render_candidate_overlay(frames[frame_idx], parent_mask, candidates, reference_mask)
plt.figure(figsize=(8, 8)); plt.imshow(preview); plt.axis('off');
plt.title(f'frame {frame_idx} | candidates={[x.candidate_id for x in candidates]}'); plt.show()


## 8. Chạy LandmarkResolver + VLM selector


In [ ]:
from quantiphy_baseline.vision.landmark_resolver import LandmarkResolver

resolver = LandmarkResolver(vlm_selector=vlm_selector)
landmark_results = resolver.resolve_plans(
    plans=[active_plan], frames=frames, fps=fps, tracks=tracks
)
for item in landmark_results:
    print('\n', item.landmark_id)
    print(' status    =', item.status)
    print(' candidate =', item.candidate_id)
    print(' method    =', item.method)
    print(' confidence=', item.confidence)
    print(' warnings  =', item.warnings)


## 9. Chạy TwoDSolver và lưu Stage 2 output


In [ ]:
from quantiphy_baseline.solver_2d import TwoDSolver

solver_results = TwoDSolver().solve_plans(
    plans=[active_plan], landmark_results=landmark_results, tracks=tracks, fps=fps
)
for item in solver_results:
    display(item.to_dict())

stage2_dir = ARTIFACT_ROOT / 'stage2_vlm' / TEST_VIDEO_ID
stage2_dir.mkdir(parents=True, exist_ok=True)
output_path = stage2_dir / f'{TEST_QA_ID}_{MODEL_PRESET}.json'
output_path.write_text(json.dumps({
    'model_id': MODEL_ID,
    'active_plan': active_plan.to_dict(),
    'landmark_results': [item.to_dict() for item in landmark_results],
    'solver_results': [item.to_dict() for item in solver_results],
}, indent=2), encoding='utf-8')
print('Saved:', output_path)


## Đọc kết quả

Kỳ vọng cho QA 1095:

1. Semantic plan: parent `pier`, landmark `outer_end`, generator `major_axis_end_caps`.
2. Candidate overlay: hai end-cap A/B; shore/reference nếu track thành công có màu cyan.
3. VLM chọn đầu phía ngoài mặt nước, không chọn đầu gắn với bờ.
4. Resolver trả `resolved` và lưu raw VLM output + overlay path.
5. Vì video là 3D, TwoDSolver hiện có thể trả `solved_pixel`; không tự bịa kết quả mét khi depth calibration chưa được triển khai.

Nếu `best_t4` OOM, restart runtime Stage 2 và đổi sang `stable_t4`; nếu vẫn OOM dùng `fast_t4`.
